In [76]:
print("="*70)
print("ZOO ANIMAL CLASSIFICATION - LAB EXAM")
print("="*70)
print(f"Roll Number: {'24UG00331'}")
print(f"Seat Number: {25}")
print(f"Method Prefix: {'Alpha'}")
print("="*70)

ZOO ANIMAL CLASSIFICATION - LAB EXAM
Roll Number: 24UG00331
Seat Number: 25
Method Prefix: Alpha


In [106]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier

In [78]:
ROLL_NUMBER = '24UG00331'
SEAT_NUMBER = 25
METHOD_PREFIX = 'Alpha'

In [79]:
class Dataexplorer_24UG00331():
    def __init__(self):
        pass

    def Alpha_load_and_integrate(self):
        pass

    def Alpha_eda_and_cleaning(self):
        pass

    def Alpha_train_and_evaluate(self):
        pass

In [80]:
explorer = Dataexplorer_24UG00331()

In [81]:
from google.colab import files

# This will open a file selection dialog. Select all 3 files:
# zoo.csv, class.csv, and auxiliary_metadata.json
uploaded = files.upload()

print("Files uploaded successfully.")

Saving auxiliary_metadata.json to auxiliary_metadata (3).json
Saving class.csv to class (3).csv
Saving zoo.csv to zoo (5).csv
Files uploaded successfully.


In [82]:
explorer.Alpha_load_and_integrate()

# --- 1. Recreate df_filtered (All Previous Steps) ---
# Load DataFrames
df_zoo = pd.read_csv("zoo.csv")
df_class = pd.read_csv("class.csv")
df_metadata = pd.read_json("auxiliary_metadata.json")

# A. Name Normalization
df_zoo['animal_name'] = df_zoo['animal_name'].str.upper()
df_metadata['animal_name'] = df_metadata['animal_name'].str.upper()
df_class = df_class.drop(columns=['Animal_Names'])

# B. df_metadata Standardization
df_metadata['conservation_status'] = df_metadata['conservation_status'].fillna(
    df_metadata['conservation']
).fillna(
    df_metadata['status']
)
df_metadata = df_metadata.drop(columns=['conservation', 'status'])
df_metadata['habitat_type'] = df_metadata['habitat'].fillna(
    df_metadata['habitats']
)
df_metadata['habitat_type'] = df_metadata['habitat_type'].replace({'fresh water': 'freshwater'})
df_metadata = df_metadata.drop(columns=['habitat', 'habitats'])
df_metadata['diet_category'] = df_metadata['diet'].fillna(
    df_metadata['diet_type']
)
df_metadata['diet_category'] = df_metadata['diet_category'].replace({'omnivor': 'omnivore'})
df_metadata = df_metadata.drop(columns=['diet', 'diet_type'])
df_metadata['diet_category'] = df_metadata['diet_category'].str.lower()
df_metadata['habitat_type'] = df_metadata['habitat_type'].str.lower()

# C. Merging
df_merged = pd.merge(df_zoo, df_class, left_on='class_type', right_on='Class_Number', how='left').drop(columns=['Class_Number'])
df_merged = pd.merge(df_merged, df_metadata, on='animal_name', how='left')

# D. Filtering
aux_cols = ['conservation_status', 'habitat_type', 'diet_category']
df_filtered = df_merged.dropna(subset=aux_cols).copy()


# --- 2. Feature Engineering ---

# Feature 1: conservation_priority
# Mapping conservation status to a numerical priority (Ordinal Encoding)
# 1 (Lowest Risk) to 4 (Highest Risk)
conservation_map = {
    'least concern': 1,
    'least': 1,               # Treating 'least' as 'least concern'
    'near threatened': 2,
    'vulnerable': 3,
    'endangered': 4
}
df_filtered['conservation_priority'] = df_filtered['conservation_status'].str.lower().map(conservation_map)

# Feature 2: aquatic_flag
# Combines binary 'aquatic' column with habitat keywords to ensure coverage.
# The '|' operator performs a logical OR on the boolean series.
df_filtered['aquatic_flag'] = (
    (df_filtered['aquatic'] == 1) |
    df_filtered['habitat_type'].str.contains('water|marine', na=False)
).astype(int)

# --- 3. Display Results ---
print("--- DataFrame Head with New Features ---")
print(df_filtered[['animal_name', 'Class_Type', 'aquatic', 'habitat_type', 'aquatic_flag', 'conservation_status', 'conservation_priority']].head(10).to_markdown(index=False, numalign="left", stralign="left"))

print("\n--- Value Counts for New Features ---")
print("conservation_priority Value Counts:")
print(df_filtered['conservation_priority'].value_counts().sort_index().to_markdown(numalign="left", stralign="left"))
print("\naquatic_flag Value Counts:")
print(df_filtered['aquatic_flag'].value_counts().sort_index().to_markdown(numalign="left", stralign="left"))


--- DataFrame Head with New Features ---
| animal_name   | Class_Type   | aquatic   | habitat_type   | aquatic_flag   | conservation_status   | conservation_priority   |
|:--------------|:-------------|:----------|:---------------|:---------------|:----------------------|:------------------------|
| AARDVARK      | Mammal       | 0         | savanna        | 0              | least concern         | 1                       |
| ANTELOPE      | Mammal       | 0         | grasslands     | 0              | near threatened       | 2                       |
| BASS          | Fish         | 1         | freshwater     | 1              | least                 | 1                       |
| BEAR          | Mammal       | 0         | forest         | 0              | vulnerable            | 3                       |
| BOAR          | Mammal       | 0         | forest         | 0              | least concern         | 1                       |
| BUFFALO       | Mammal       | 0         | grasslands 

In [84]:
explorer.Alpha_eda_and_cleaning()
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 6)

# 1. Pie Chart: Class Distribution
plt.figure(figsize=(8, 8))
class_counts = df_filtered['Class_Type'].value_counts()
plt.pie(
    class_counts,
    labels=class_counts.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=sns.color_palette("pastel")
)
plt.title('Distribution of Animal Classes (N=10)', fontsize=16)
plt.ylabel('') # Hide the default y-label
plt.tight_layout()
plt.savefig('class_distribution_pie_chart.png')
plt.close()
print("Generated: class_distribution_pie_chart.png")


# 2. Distribution Plot: Legs/Fins Distribution (Using a Box Plot due to N=10)
# A simple bar plot showing the non-zero counts for legs/fins is more informative for N=10.
df_lf = df_filtered[df_filtered[['legs', 'fins']].sum(axis=1) > 0].copy()
df_lf_melt = df_lf.melt(id_vars='animal_name', value_vars=['legs', 'fins'], var_name='Feature', value_name='Value')

plt.figure(figsize=(10, 6))
# Using a boxenplot (a good distribution summary plot) to visualize the minimal variability
sns.boxenplot(data=df_lf_melt, x='Feature', y='Value', hue='Feature', palette="Set2", showfliers=False)
plt.title('Distribution of Non-Zero Legs and Fins Values (N=10)', fontsize=16)
plt.xlabel('Feature', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.tight_layout()
plt.savefig('legs_fins_distribution_boxenplot.png')
plt.close()
print("Generated: legs_fins_distribution_boxenplot.png")


# 3. Count Plot: Habitat Types by Diet
plt.figure(figsize=(10, 7))
sns.countplot(
    data=df_filtered,
    y='habitat_type',
    hue='diet_category',
    order=df_filtered['habitat_type'].value_counts().index, # Order by count
    palette="viridis"
)
plt.title('Count of Habitat Types by Diet Category (N=10)', fontsize=16)
plt.xlabel('Count', fontsize=12)
plt.ylabel('Habitat Type', fontsize=12)
plt.tight_layout()
plt.savefig('habitat_by_diet_countplot.png')
plt.close()
print("Generated: habitat_by_diet_countplot.png")


# 4. Annotated Heatmap: Feature Correlations
# Select relevant numerical and engineered features
numerical_features = ['hair', 'eggs', 'milk', 'aquatic', 'legs', 'fins', 'catsize', 'conservation_priority', 'aquatic_flag']
corr_matrix = df_filtered[numerical_features].corr()

plt.figure(figsize=(10, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=.5,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
plt.title('Annotated Feature Correlation Heatmap (N=10)', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('feature_correlation_heatmap.png')
plt.close()
print("Generated: feature_correlation_heatmap.png")

print("\nAll visualizations have been generated and saved.")

Generated: class_distribution_pie_chart.png
Generated: legs_fins_distribution_boxenplot.png
Generated: habitat_by_diet_countplot.png
Generated: feature_correlation_heatmap.png

All visualizations have been generated and saved.


In [86]:
# A. Class Imbalance Ratio
class_counts = df_filtered['Class_Type'].value_counts()
largest_class_size = class_counts.max()
smallest_class_size = class_counts.min()

if smallest_class_size > 0:
    imbalance_ratio = largest_class_size / smallest_class_size
else:
    imbalance_ratio = np.inf

print("--- 1. Class Imbalance Ratio ---")
print(f"Largest Class Size: {largest_class_size}")
print(f"Smallest Class Size: {smallest_class_size}")
print(f"Imbalance Ratio (Largest/Smallest): {imbalance_ratio:.2f}")


# B. Low Variance Features (Variance < 0.01)
# Selecting all binary, engineered, and general quantitative features
numerical_features = ['hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator', 'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs', 'tail', 'domestic', 'catsize', 'conservation_priority', 'aquatic_flag']
df_numerical = df_filtered[numerical_features]

variances = df_numerical.var()
low_variance_features = variances[variances < 0.01]

print("\n--- 2. Low Variance Features (Variance < 0.01) ---")
if not low_variance_features.empty:
    print("Features with variance < 0.01:")
    print(low_variance_features.to_markdown(numalign="left", stralign="left"))
else:
    print("No numerical features found with variance less than 0.01.")


# C. Highly Correlated Pairs (> 0.8)
corr_matrix = df_numerical.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
highly_correlated_pairs = upper.unstack()
highly_correlated_pairs = highly_correlated_pairs[highly_correlated_pairs > 0.8].sort_values(ascending=False)

print("\n--- 3. Highly Correlated Pairs (> 0.8) ---")
if not highly_correlated_pairs.empty:
    print("Highly Correlated Feature Pairs (|Correlation| > 0.8):")
    # Format the output as a readable list
    output = []
    for (feature1, feature2), correlation in highly_correlated_pairs.items():
        # Retrieve the original signed correlation value for accurate reporting
        original_corr = df_numerical[[feature1, feature2]].corr().iloc[0, 1]
        output.append(f"({feature1} & {feature2}): {original_corr:.3f}")

    print("\n".join(output))
else:
    print("No highly correlated feature pairs found with |Correlation| > 0.8.")

--- 1. Class Imbalance Ratio ---
Largest Class Size: 6
Smallest Class Size: 2
Imbalance Ratio (Largest/Smallest): 3.00

--- 2. Low Variance Features (Variance < 0.01) ---
Features with variance < 0.01:
|          | 0   |
|:---------|:----|
| feathers | 0   |
| airborne | 0   |
| venomous | 0   |

--- 3. Highly Correlated Pairs (> 0.8) ---
Highly Correlated Feature Pairs (|Correlation| > 0.8):
(eggs & hair): -1.000
(milk & hair): 1.000
(milk & eggs): -1.000
(breathes & hair): 1.000
(breathes & milk): 1.000
(backbone & toothed): 1.000
(breathes & eggs): -1.000
(aquatic_flag & breathes): -1.000
(catsize & eggs): -1.000
(catsize & milk): 1.000
(catsize & breathes): 1.000
(catsize & hair): 1.000
(aquatic_flag & catsize): -1.000
(aquatic_flag & milk): -1.000
(aquatic_flag & eggs): 1.000
(aquatic_flag & hair): -1.000
(aquatic & eggs): 0.802
(aquatic_flag & aquatic): 0.802
(aquatic & milk): -0.802
(aquatic & hair): -0.802
(aquatic_flag & legs): -0.802
(breathes & aquatic): -0.802
(catsize & le

In [96]:
explorer.Alpha_train_and_evaluate()


feature_cols = [
    'hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic', 'predator',
    'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs', 'tail',
    'domestic', 'catsize', 'conservation_priority', 'aquatic_flag'
]
X = df_filtered[feature_cols]

# Target (Class Type)
y = df_filtered['Class_Type']


# --- 3. Train/Test Split ---

# Train=75%, Test=25%, random_state=123
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=123
)

# --- 4. Display Results ---

print("--- Split Summary ---")
print(f"Total Samples (N): {len(df_filtered)}")
print(f"Training Set Size (X_train, y_train): {len(X_train)} samples")
print(f"Testing Set Size (X_test, y_test): {len(X_test)} samples")
print(f"Actual Train Percentage: {len(X_train)/len(df_filtered)*100:.1f}%")

print("\n--- Training Class Distribution ---")
print(y_train.value_counts().to_markdown(numalign="left", stralign="left"))

print("\n--- Testing Class Distribution ---")
print(y_test.value_counts().to_markdown(numalign="left", stralign="left"))

print("\n--- Feature Matrix (X) Head ---")
print(X_train.head().to_markdown(index=True, numalign="left", stralign="left"))

--- Split Summary ---
Total Samples (N): 10
Training Set Size (X_train, y_train): 7 samples
Testing Set Size (X_test, y_test): 3 samples
Actual Train Percentage: 70.0%

--- Training Class Distribution ---
| Class_Type   | count   |
|:-------------|:--------|
| Mammal       | 4       |
| Fish         | 2       |
| Invertebrate | 1       |

--- Testing Class Distribution ---
| Class_Type   | count   |
|:-------------|:--------|
| Mammal       | 2       |
| Invertebrate | 1       |

--- Feature Matrix (X) Head ---
|    | hair   | feathers   | eggs   | milk   | airborne   | aquatic   | predator   | toothed   | backbone   | breathes   | venomous   | fins   | legs   | tail   | domestic   | catsize   | conservation_priority   | aquatic_flag   |
|:---|:-------|:-----------|:-------|:-------|:-----------|:----------|:-----------|:----------|:-----------|:-----------|:-----------|:-------|:-------|:-------|:-----------|:----------|:------------------------|:---------------|
| 5  | 1      | 0    

In [110]:

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_split=2,
    random_state=123 # Keep this for reproducibility
)

# Training
rf_model.fit(X_train, y_train)


# --- 3. Evaluation ---

# Predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Scores
train_accuracy = accuracy_score(y_train, y_train_pred)
rf_test_accuracy = accuracy_score(y_test, y_test_pred) # Storing RF test accuracy

print("--- Random Forest Model Training and Evaluation ---")
print(f"n_estimators: {rf_model.n_estimators}")
print(f"max_depth: {rf_model.max_depth}")
print(f"min_samples_split: {rf_model.min_samples_split}")
print("-" * 50)
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy (N=3): {rf_test_accuracy:.4f}")
print(f"overfitting gap: {train_accuracy-rf_test_accuracy:.4f}")

# Generate the report as a dictionary for easier parsing
report_dict = classification_report(y_test, y_test_pred, output_dict=True)
report_str = classification_report(y_test, y_test_pred)

print("--- Random Forest Classification Report on Test Set (N=3) ---")
print(report_str)

feature_importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(12) # Top 12

# Identify engineered features for highlighting
engineered_features = ['conservation_priority', 'aquatic_flag']

# Create color list: highlight engineered features (orange) vs original features (blue)
colors = ['#FF7F0E' if feature in engineered_features else '#1F77B4' for feature in feature_importances['Feature']]

plt.figure(figsize=(10, 7))
sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importances,
    palette=colors,
    orient='h'
)

plt.title('Top 12 Feature Importances from Random Forest Classifier', fontsize=16)
plt.xlabel('Feature Importance (Mean Decrease in Impurity)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.gca().invert_yaxis() # Highest importance at the top

# Add a legend for highlighting
import matplotlib.patches as mpatches
blue_patch = mpatches.Patch(color='#1F77B4', label='Original Feature')
orange_patch = mpatches.Patch(color='#FF7F0E', label='Engineered Feature')
plt.legend(handles=[orange_patch, blue_patch], title='Feature Type')

plt.tight_layout()
plt.savefig('top_12_feature_importances_bar_chart.png')
plt.close()
print("Generated: top_12_feature_importances_bar_chart.png")

# --- 3. Regenerate Heatmap Code (Ensuring Labeled Axes and Title) ---
# Select numerical features for correlation plot (excluding low variance ones for clarity)
numerical_features_corr = [
    'hair', 'eggs', 'milk', 'aquatic', 'predator', 'toothed', 'backbone',
    'breathes', 'fins', 'legs', 'tail', 'catsize', 'conservation_priority',
    'aquatic_flag'
]
corr_matrix = df_filtered[numerical_features_corr].corr()

sns.set_style("white")
plt.figure(figsize=(12, 11))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=.5,
    cbar_kws={'label': 'Pearson Correlation Coefficient'}
)
plt.title(f'Annotated Feature Correlation Heatmap (N={len(df_filtered)})', fontsize=16)
plt.xlabel('Features', fontsize=14)
plt.ylabel('Features', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig('annotated_correlation_heatmap_labeled.png')
plt.close()
print("Generated: annotated_correlation_heatmap_labeled.png")

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

# Training
knn_model.fit(X_train, y_train)


# --- 3. Evaluation ---

# Predictions
y_train_pred_knn = knn_model.predict(X_train)
y_test_pred_knn = knn_model.predict(X_test)

# Scores
train_accuracy_knn = accuracy_score(y_train, y_train_pred_knn)
k_nn_test_accuracy = accuracy_score(y_test, y_test_pred_knn) # Storing KNN test accuracy

print("--- K-Nearest Neighbors Model Training and Evaluation ---")
print(f"n_neighbors (k): {knn_model.n_neighbors}")
print("-" * 50)
print(f"Training Accuracy: {train_accuracy_knn:.4f}")
print(f"Testing Accuracy (N=3): {k_nn_test_accuracy:.4f}")

# --- Populate MODEL ANALYSIS variables ---

# 1. Most important feature
top_feature = feature_importances.iloc[0]['Feature']
top_value = feature_importances.iloc[0]['Importance']

# 2 & 3. Worst and Best performing class (F1-score)
class_f1_scores = {k: v['f1-score'] for k, v in report_dict.items() if k not in ['accuracy', 'macro avg', 'weighted avg']}

if class_f1_scores:
    worst_class = min(class_f1_scores, key=class_f1_scores.get)
    worst_f1 = class_f1_scores[worst_class]
    best_class = max(class_f1_scores, key=class_f1_scores.get)
    best_f1 = class_f1_scores[best_class]
else:
    worst_class = "N/A"
    worst_f1 = 0.0
    best_class = "N/A"
    best_f1 = 0.0

# 4. Engineered feature rank
highest_ranked_engineered_feature = None
highest_ranked_engineered_feature_rank = float('inf')

for ef in engineered_features:
    if ef in feature_importances['Feature'].values:
        # Get the rank (index + 1) of the engineered feature
        rank = feature_importances[feature_importances['Feature'] == ef].index[0] + 1
        if rank < highest_ranked_engineered_feature_rank:
            highest_ranked_engineered_feature_rank = rank
            highest_ranked_engineered_feature = ef

engineered_feature_name = highest_ranked_engineered_feature if highest_ranked_engineered_feature else "N/A"
engineered_feature_rank = highest_ranked_engineered_feature_rank if highest_ranked_engineered_feature else "N/A"


# 5. Model comparison
comparison_model_name = "KNN"
comparison_accuracy = k_nn_test_accuracy
rf_acc = rf_test_accuracy


print("\n=== MODEL ANALYSIS ===")
print(f"1. most important feature:{top_feature}(importance:{top_value:.3f})")
print(f"2. worst performing class:{worst_class}(f1:{worst_f1:.3f})")
print(f"3. best performing class:{best_class}(f1:{best_f1:.3f})")
print(f"4. engineered feature:{engineered_feature_name} ranked #{engineered_feature_rank}")
print(f"5. model comparison:{comparison_model_name}={comparison_accuracy:.3f} vs RF={rf_acc:.3f}")

--- Random Forest Model Training and Evaluation ---
n_estimators: 150
max_depth: 15
min_samples_split: 2
--------------------------------------------------
Training Accuracy: 1.0000
Testing Accuracy (N=3): 1.0000
overfitting gap: 0.0000
--- Random Forest Classification Report on Test Set (N=3) ---
              precision    recall  f1-score   support

Invertebrate       1.00      1.00      1.00         1
      Mammal       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



/tmp/ipython-input-3704965989.py:50: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


Generated: top_12_feature_importances_bar_chart.png
Generated: annotated_correlation_heatmap_labeled.png
--- K-Nearest Neighbors Model Training and Evaluation ---
n_neighbors (k): 5
--------------------------------------------------
Training Accuracy: 0.8571
Testing Accuracy (N=3): 0.6667

=== MODEL ANALYSIS ===
1. most important feature:milk(importance:0.121)
2. worst performing class:Invertebrate(f1:1.000)
3. best performing class:Invertebrate(f1:1.000)
4. engineered feature:conservation_priority ranked #17
5. model comparison:KNN=0.667 vs RF=1.000
